# Measurement before rankings: Combine-to-game evidence

Development-only analysis of 2023/2024 official data. Rebuilds eligibility, measurements, cross-cohort fits, game-label checks and shrinkage from attached competition files. Six figures; 2025 remains reserved. Null findings and uncertainty are part of the result. No official data rows are embedded in source or exported as CSV.

Data credit: NFL Big Data Bowl 2027 competition organizers. Use the attached data under the competition rules and applicable data license. The code and analysis do not confer redistribution rights over the underlying data.

In [ ]:

from pathlib import Path
import base64,gzip,hashlib,json,shutil,subprocess,sys,time
import numpy as np,pandas as pd
from IPython.display import display,Markdown,Image
assets=json.loads(gzip.decompress(base64.b64decode('...')))
project=Path('/kaggle/working/research');project.mkdir();data=project/'data';data.mkdir()
for name,source in assets['sources'].items():(project/name).write_text(source,encoding='utf-8')
# Use exact official floating-point inputs without an intermediate CSV rewrite.
# Metadata audits receive only development rows in memory. The unchanged downstream
# analysis scripts discard held-out rows before calculations, as in prior verification.
for name in ['players.csv','combine_results.csv','games.csv','combine_tracking.csv','player_play.csv']:
    matches=list(Path('/kaggle/input').rglob(name));assert len(matches)==1
    (data/name).symlink_to(matches[0])
players=pd.read_csv(data/'players.csv',usecols=['nfl_id','draft_year'])
ids=set(players.loc[players.draft_year.isin([2023,2024]),'nfl_id'])
games=pd.read_csv(data/'games.csv',usecols=['game_id','season'])
game_ids=set(games.loc[games.season.isin([2023,2024]),'game_id'])
sys.path.insert(0,str(project))
from audit_tracking import audit_tracking
from audit_player_play import audit_player_play
from run_feasibility import link_coverage
meta=project/'artifacts/feasibility_v1'
started=time.time();original_read_csv=pd.read_csv
def development_read_csv(path,*args,**kwargs):
    frame=original_read_csv(path,*args,**kwargs)
    if Path(path).parent!=data:return frame
    assert isinstance(frame,pd.DataFrame),'Metadata audit expects nonchunked reads'
    name=Path(path).name
    if name in ['players.csv','combine_results.csv','combine_tracking.csv']:return frame.loc[frame.draft_year.isin([2023,2024])].copy()
    if name=='games.csv':return frame.loc[frame.season.isin([2023,2024])].copy()
    if name=='player_play.csv':return frame.loc[frame.nfl_id.isin(ids)&frame.game_id.isin(game_ids)].copy()
    raise AssertionError(name)
try:
    pd.read_csv=development_read_csv
    audit_tracking(data,meta);audit_player_play(data,meta);link_coverage(meta)
finally:pd.read_csv=original_read_csv
assert pd.read_csv(meta/'player_split.csv').subset.eq('development').all()
(project/'measurement_before_rankings_draft_en.md').write_text(assets['manuscript'],encoding='utf-8')
for name in ['measure_reliability.py','check_measurement_sensitivity.py','run_development_transfer.py','check_outcome_reliability.py','run_shrinkage_validation.py']:
    subprocess.run([sys.executable,str(project/name)],check=True,cwd=project,timeout=1500)
verified=[]
for name,expected in assets['checks'].items():
    frame=pd.read_csv(project/'artifacts'/name)
    assert len(frame)==expected['rows'] and list(frame)==expected['columns'],name
    for col,c in expected['checks'].items():
        s=frame[col]
        if c['float']:
            assert s.isna().sum()==c['missing'],(name,col,'missing')
            a=s.dropna().to_numpy()
            actual={'sum':float(a.sum()),'mean':float(a.mean()) if len(a) else None,'min':float(a.min()) if len(a) else None,'max':float(a.max()) if len(a) else None,'sum_squares':float(a@a)}
            for k,v in actual.items():
                assert (v is None and c[k] is None) or np.isclose(v,c[k],atol=max(1,len(a))*1e-4,rtol=1e-6),(name,col,k,v,c[k])
        else:assert hashlib.sha256(s.to_json(orient='values').encode()).hexdigest()==c['sha256'],(name,col)
    verified.append(name)
# Recompute conditional uncertainty from this runtime's predictions. Minor numerical
# solver differences must not pair an old point estimate with new predictions.
predictions=pd.read_csv(project/'artifacts/development_transfer_v1/oof_predictions.csv')
errors=predictions.assign(error=abs(predictions.outcome-predictions.prediction)).pivot(index=['validation_year','nfl_id'],columns='model',values='error')
comparisons=[]
for name in ['plus_gauntlet_speed','plus_gauntlet_lateral']:
    delta=errors[name]-errors.traditional;rng=np.random.default_rng(20261018)
    parts=[g.to_numpy() for _,g in delta.groupby(level='validation_year')]
    draws=np.concatenate([a[rng.integers(0,len(a),size=(5000,len(a)))] for a in parts],axis=1).mean(axis=1)
    comparisons.append({'model':name,'mae_delta_vs_traditional':float(delta.mean()),'stratified_player_bootstrap_95':np.quantile(draws,[.025,.975]).tolist(),'negative_means_improvement':True})
prior={'passed':True,'players':37,'routes':8326,'holdout_excluded':True,'comparisons':comparisons,
    'bootstrap_seed':20261018,'bootstrap_draws':5000,'interval_provenance':'Recomputed year-stratified paired-player bootstrap from this runtime fitted predictions; conditional, no training refit or selection correction.'}
(project/'artifacts/development_transfer_v1/verification.json').write_text(json.dumps(prior,indent=2))
manuscript=assets['manuscript'].replace('It has not been published or submitted.', 'This notebook reports the development analysis; no 2025 confirmation is claimed.')
scores=pd.read_csv(project/'artifacts/development_transfer_v1/metrics.csv');scores=scores.loc[scores.validation_year.eq('pooled')].set_index('model')
for name,old in [('traditional','0.22345'),('plus_gauntlet_speed','0.21875'),('plus_gauntlet_lateral','0.23476')]:manuscript=manuscript.replace(old,f"{scores.loc[name,'mae']:.5f}")
for comparison,old in zip(comparisons,['[-0.01697, +0.00755]','[+0.00119, +0.02131]']):
    lo,hi=comparison['stratified_player_bootstrap_95'];manuscript=manuscript.replace(old,f'[{lo:+.5f}, {hi:+.5f}]')
shrink=json.loads((project/'artifacts/shrinkage_v1/summary.json').read_text())
metrics={x['model']:x for x in shrink['metrics'] if x['outcome']=='context_adjusted'}
for name,old in [('source_half_mean','0.27097'),('shrunken_mean','0.24836')]:manuscript=manuscript.replace(old,f"{metrics[name]['mae']:.5f}")
lo,hi=next(x for x in shrink['comparisons'] if x['outcome']=='context_adjusted')['paired_player_bootstrap_95']
manuscript=manuscript.replace('[-0.05959, +0.01476]',f'[{lo:+.5f}, {hi:+.5f}]')
(project/'measurement_before_rankings_draft_en.md').write_text(manuscript,encoding='utf-8')
subprocess.run([sys.executable,str(project/'build_manuscript_figures.py')],check=True,cwd=project,timeout=180)
figures=project/'artifacts/manuscript_figures_v1';manifest=json.loads((figures/'manifest.json').read_text())
assert len(manifest['figures'])==6 and manifest['word_count_whitespace']<2000
release=Path('/kaggle/working/release');release.mkdir()
for item in manifest['figures']:shutil.copy2(figures/(item['name']+'.png'),release/(item['name']+'.png'))
manuscript=(project/'measurement_before_rankings_illustrated_en.md').read_text()
manuscript=manuscript.replace('Six figures are included; a public reproducible notebook and final confirmation are still required.','Six figures and the complete development reproduction are included in this notebook. The 2025 cohort remains reserved.')
manuscript=manuscript.replace('Numerical provenance: archived development analyses and `artifacts/manuscript_evidence_v1/evidence.json`. This illustrated draft remains unpublished.','Numerical provenance: development analyses reproduced below from the attached official competition files. Only 2023 and 2024 enter these calculations. The 2025 cohort remains reserved.')
manuscript=manuscript.replace('artifacts/manuscript_figures_v1/','')
(release/'manuscript.md').write_text(manuscript)
(release/'prediction_intervals.json').write_text(json.dumps(prior,indent=2))
allowed=['reliability_v1/reliability.csv','measurement_sensitivity_v1/reliability.csv','development_transfer_v1/metrics.csv','development_transfer_v1/context_diagnostics.csv','outcome_reliability_v1/reliability.csv','shrinkage_v1/metrics.csv']
for name in allowed:shutil.copy2(project/'artifacts'/name,release/(name.replace('/','_')))
receipt={'state':'complete','utc':__import__('datetime').datetime.now(__import__('datetime').timezone.utc).isoformat(),'seconds':time.time()-started,
    'source_sha256':{n:hashlib.sha256(s.encode()).hexdigest() for n,s in assets['sources'].items()},
    'checked_tables':verified,'check_method':'Exact nonfloating column hashes; numeric column moments and extrema within documented tolerance. Not rowwise floating equivalence.',
    'figures':6,'official_inputs':True,'embedded_official_rows':False,'metadata_recomputed_from_official_inputs':True,'development_years':[2023,2024],
    'versions':{'python':sys.version,'numpy':np.__version__,'pandas':pd.__version__,'sklearn':__import__('sklearn').__version__},
    'holdout_evaluated':False,'formal_writeup_submitted':False,'interval_provenance':prior['interval_provenance'],
    'output_policy':'Only prose, figures and six aggregate tables exported; all raw/individual intermediate files removed.'}
(release/'validation.json').write_text(json.dumps(receipt,indent=2))
display(Markdown(manuscript))
for item in manifest['figures']:
    display(Image(filename=str(release/(item['name']+'.png'))));display(Markdown(item['caption']))
# The public output must not contain symlinks to raw data or per-player/per-frame CSVs.
shutil.rmtree(project)
assert not project.exists() and all(p.suffix in ['.csv','.json','.md','.png'] for p in release.iterdir())
print(json.dumps(receipt),flush=True)
